# Final results — policy improvement over iterations

Consumes two JSON artifacts:
- `results/checkpoint_eval.json` — output of `training/evaluate.py` with per-episode records
- `results/fle_cross_check.json` — output of `translator/run_topk_cross_check.py`

Produces:
1. Per-checkpoint raw-metrics table (plan.md §Reward reporting).
2. Composite-reward-vs-iteration plot.
3. Per-iteration delta bar chart (headline claim: `Δ_final > max_i Δ_i`).
4. Before/after visualization for the single best layout in `policy_final` vs its counterpart in `policy_0`.
5. FLE cross-check summary: build success rate, Pearson r, MAPE, ship-gate status.
6. Scatter of sim rate vs FLE rate with y=x reference.

## 0. Setup

In [ ]:
import json, pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

EVAL_JSON = pathlib.Path('../results/checkpoint_eval.json')
FLE_JSON = pathlib.Path('../results/fle_cross_check.json')

eval_data = json.loads(EVAL_JSON.read_text()) if EVAL_JSON.exists() else []
fle_data = json.loads(FLE_JSON.read_text()) if FLE_JSON.exists() else {}

checkpoint_order = ['policy_0', 'policy_1', 'policy_2', 'policy_3', 'policy_final']
print(f'checkpoints found: {[c["name"] for c in eval_data]}')
print(f'fle checkpoints:   {list(fle_data.keys())}')

## 1. Per-checkpoint raw-metrics table

Plan §Reward reporting: green-science rate, materials, cells (area), machines, valid-output rate. Composite is shown alongside as a summary column.

In [ ]:
rows = []
for c in eval_data:
    rows.append({
        'checkpoint': c['name'],
        'green_science/s': round(c['mean_green_science'], 4),
        'materials': round(c['mean_materials'], 1),
        'cells (area)': round(c['mean_cells'], 1),
        'machines': round(c['mean_machines'], 2),
        'valid %': round(c['valid_output_pct'], 1),
        'parse ok %': round(c['parse_ok_pct'], 1),
        'composite': round(c['mean_composite'], 4),
        'n samples': c['n_samples'],
    })
df = pd.DataFrame(rows)
if not df.empty:
    df = df.set_index('checkpoint').reindex([n for n in checkpoint_order if n in df.index])
df

## 2. Composite reward vs iteration

In [ ]:
if eval_data:
    names = [c['name'] for c in sorted(eval_data, key=lambda c: checkpoint_order.index(c['name']) if c['name'] in checkpoint_order else 99)]
    composites = [c['mean_composite'] for c in sorted(eval_data, key=lambda c: checkpoint_order.index(c['name']) if c['name'] in checkpoint_order else 99)]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(names, composites, marker='o')
    ax.set_xlabel('Checkpoint')
    ax.set_ylabel('Mean composite reward (val)')
    ax.set_title('Composite reward vs iteration')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. Per-iteration deltas

Task 3 claim: `Δ_final > max_i Δ_i` where `Δ_i = mean(policy_{i+1}) - mean(policy_i)`.

Verified if the bar for `Δ_final` (last vs first-checkpoint delta) exceeds every intermediate step delta.

In [ ]:
if len(eval_data) >= 2:
    ordered = sorted(eval_data, key=lambda c: checkpoint_order.index(c['name']) if c['name'] in checkpoint_order else 99)
    step_deltas = [ordered[i+1]['mean_composite'] - ordered[i]['mean_composite'] for i in range(len(ordered)-1)]
    step_labels = [f"{ordered[i]['name']}→{ordered[i+1]['name']}" for i in range(len(ordered)-1)]
    total_delta = ordered[-1]['mean_composite'] - ordered[0]['mean_composite']

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(step_labels, step_deltas, color='steelblue', label='per-iteration')
    ax.axhline(total_delta, color='crimson', linestyle='--', label=f'Δ_final = {total_delta:+.4f}')
    ax.set_ylabel('Δ composite reward')
    ax.set_title('Per-iteration deltas vs Δ_final')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

    max_step = max(step_deltas) if step_deltas else 0.0
    print(f'\nΔ_final = {total_delta:+.4f}')
    print(f'max per-iteration Δ = {max_step:+.4f}')
    print(f'CLAIM (Δ_final > max_i Δ_i): {"PASS" if total_delta > max_step else "FAIL"}')

## 4. Best-layout visualization — policy_0 vs policy_final

Pick the highest-composite episode from each checkpoint (from `episodes[]`) and print entity counts + JSON snippet. A future improvement adds a matplotlib grid renderer.

In [ ]:
def best_episode(ckpt):
    eps = [e for e in (ckpt.get('episodes') or []) if e['parse_ok']]
    if not eps:
        return None
    return max(eps, key=lambda e: (e['composite'], e['green_science_rate']))

for name in ('policy_0', 'policy_final'):
    ckpt = next((c for c in eval_data if c['name'] == name), None)
    if not ckpt:
        continue
    ep = best_episode(ckpt)
    if ep is None:
        print(f'{name}: no parse-ok episodes')
        continue
    from mini_factorio.layout import Layout
    lay = Layout.model_validate_json(ep['layout_after_json'])
    print(f'--- {name}: best episode (layout_index={ep["layout_index"]}) ---')
    print(f'  composite={ep["composite"]:.4f}  gs={ep["green_science_rate"]:.4f}/s')
    print(f'  entities: machines={len(lay.machines)} inserters={len(lay.inserters)} belts={len(lay.belts)}')
    print(f'  materials={ep["materials"]:.0f}  cells={ep["cells"]}\n')

## 5. FLE cross-check summary

In [ ]:
fle_rows = []
for name, rep in fle_data.items():
    fle_rows.append({
        'checkpoint': name,
        'build success': f"{rep['build_success_rate']:.0%}",
        'Pearson r': None if rep.get('pearson_r') is None else round(rep['pearson_r'], 3),
        'MAPE': None if rep.get('mape') is None else f"{rep['mape']:.1%}",
        'gate: build_100': rep['ship_gates']['build_success_100pct'],
        'gate: r>=0.9': rep['ship_gates']['pearson_r_ge_0_9'],
        'gate: MAPE<=20%': rep['ship_gates']['mape_le_0_20'],
    })
pd.DataFrame(fle_rows).set_index('checkpoint') if fle_rows else 'no FLE data'

## 6. sim rate vs FLE rate scatter

In [ ]:
sim_all, fle_all, labels = [], [], []
for name, rep in fle_data.items():
    for r in rep['per_layout']:
        if r.get('build_ok') and r.get('fle_rate') is not None and r.get('sim_rate') is not None:
            sim_all.append(r['sim_rate'])
            fle_all.append(r['fle_rate'])
            labels.append(name)
if sim_all:
    fig, ax = plt.subplots(figsize=(6, 6))
    for n in set(labels):
        xs = [s for s, l in zip(sim_all, labels) if l == n]
        ys = [f for f, l in zip(fle_all, labels) if l == n]
        ax.scatter(xs, ys, label=n)
    lo = min(min(sim_all), min(fle_all))
    hi = max(max(sim_all), max(fle_all))
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.4, label='y=x')
    ax.set_xlabel('sim rate (items/s)')
    ax.set_ylabel('FLE rate (items/s)')
    ax.set_title('Simulator vs FLE — per-layout rates')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()